# Chapter 6: Object References, Mutability, and Recycling
*Fluent Python, 2nd Edition -- Luciano Ramalho*

> "Variables are labels, not boxes."

This notebook is an interactive companion to Chapter 6. Run every cell -- experiment, break things, and build intuition about how Python manages objects in memory.

**Concepts covered:**
1. Variables are labels (sticky notes on objects)
2. Identity vs equality (`==` vs `is`)
3. Aliases
4. Relative immutability of tuples
5. Shallow vs deep copies
6. Function parameters as references (call by sharing)
7. Mutable default parameters trap
8. Defensive programming with mutable params
9. `del` and garbage collection
10. Interning tricks with immutables

## 1. Variables Are Labels, Not Boxes

In Python, a variable is a **name** (label/sticky note) attached to an object -- not a box containing a value. Assignment binds a name to an object that already exists on the right-hand side.

In [ ]:
# Two labels on the SAME list object
a = [1, 2, 3]
b = a            # b is bound to the same object as a

a.append(4)
print("a =", a)
print("b =", b)   # b sees the change -- same object!
print("a is b:", a is b)

In [ ]:
# The right-hand side is evaluated FIRST, then bound
class Gizmo:
    def __init__(self):
        print(f"  Gizmo created: id={id(self)}")

x = Gizmo()       # object created, then bound to x

try:
    y = Gizmo() * 10   # Gizmo created, but * fails -> y never bound
except TypeError as e:
    print(f"  Error: {e}")

print("'x' in dir():", 'x' in dir())
print("'y' in dir():", 'y' in dir())  # y was never assigned

## 2. Identity, Equality, and Aliases

| Operator | Tests | Invokes |
|----------|-------|--------|
| `==` | **equality** (same value?) | `__eq__()` |
| `is` | **identity** (same object?) | compares `id()` values |

Two names bound to the same object are **aliases**.

In [ ]:
# Aliases: charles and lewis point to the SAME dict
charles = {"name": "Charles L. Dodgson", "born": 1832}
lewis = charles                          # alias

print("lewis is charles:", lewis is charles)
print("id(charles):", id(charles))
print("id(lewis):  ", id(lewis))

lewis["pen_name"] = "Lewis Carroll"      # mutating via one alias...
print("charles:", charles)               # ...visible through the other

In [ ]:
# Equal but NOT identical
alex = {"name": "Charles L. Dodgson", "born": 1832, "pen_name": "Lewis Carroll"}
print("alex == charles:", alex == charles)    # same value
print("alex is charles:", alex is charles)    # different objects
print("alex is not charles:", alex is not charles)  # Pythonic negation

## 3. Choosing Between `==` and `is`

**Rule of thumb:** Use `==` almost everywhere. Use `is` **only** for singletons (especially `None`).

```python
if x is None:       # correct
if x is not None:   # correct
if x == None:       # works but slower and can be overridden
```

`is` is faster (just compares two integer IDs), but `==` is what you actually *mean* most of the time.

In [ ]:
# Sentinel objects -- a valid use-case for "is"
END_OF_DATA = object()   # unique sentinel

data = [1, 2, 3, END_OF_DATA, 4]

for item in data:
    if item is END_OF_DATA:
        print("-> Reached sentinel, stopping.")
        break
    print(f"   Processing: {item}")

## 4. The Relative Immutability of Tuples

A tuple's **structure** (which references it holds) is immutable, but if those references point to **mutable objects**, those objects can still change.

In [ ]:
t1 = (1, 2, [30, 40])
t2 = (1, 2, [30, 40])

print("Before mutation:")
print("  t1 == t2:", t1 == t2)      # equal values

# Mutate the list INSIDE t1
t1[-1].append(99)
print("\nAfter t1[-1].append(99):")
print("  t1:", t1)                   # (1, 2, [30, 40, 99])
print("  t2:", t2)                   # (1, 2, [30, 40])
print("  t1 == t2:", t1 == t2)      # no longer equal!

# But the tuple "container" is still immutable:
try:
    t1[0] = 99
except TypeError as e:
    print(f"\n  Cannot assign to tuple item: {e}")

## 5. Shallow vs Deep Copies

| Method | Depth | Shared inner objects? |
|--------|-------|----------------------|
| `list(x)`, `x[:]`, `copy.copy(x)` | **Shallow** | Yes |
| `copy.deepcopy(x)` | **Deep** | No (recursively cloned) |

In [ ]:
# Shallow copy demo
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)          # shallow copy

print("l1 is l2:", l1 is l2)        # different list objects
print("l1[1] is l2[1]:", l1[1] is l2[1])  # but SHARE inner list

l1.append(100)          # only affects l1
l1[1].remove(55)        # affects BOTH (shared inner list!)

print("\nl1:", l1)
print("l2:", l2)        # l2[1] also lost 55

# += on mutable (list) -> in-place mutation (shared)
l2[1] += [33, 22]
# += on immutable (tuple) -> creates new object (not shared)
l2[2] += (10, 11)

print("\nAfter += operations:")
print("l1:", l1)
print("l2:", l2)
print("l1[1] is l2[1]:", l1[1] is l2[1])  # still the same list
print("l1[2] is l2[2]:", l1[2] is l2[2])  # different tuples now

In [ ]:
import copy

class Bus:
    """School bus: picks up and drops off passengers."""
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = list(passengers)  # defensive copy!

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

    def __repr__(self):
        return f"Bus({self.passengers!r})"

bus1 = Bus(["Alice", "Bill", "Claire", "David"])
bus2 = copy.copy(bus1)      # shallow
bus3 = copy.deepcopy(bus1)  # deep

bus1.drop("Bill")

print("bus1 (original): ", bus1)
print("bus2 (shallow):  ", bus2)   # Bill is gone here too!
print("bus3 (deep):     ", bus3)   # Bill is still here

print("\nbus1.passengers is bus2.passengers:", bus1.passengers is bus2.passengers)
print("bus1.passengers is bus3.passengers:", bus1.passengers is bus3.passengers)

In [ ]:
# deepcopy handles cyclic references gracefully
a = [10, 20]
b = [a, 30]
a.append(b)    # a -> b -> a  (cycle!)

print("a:", a)  # [10, 20, [[...], 30]]

c = copy.deepcopy(a)
print("c:", c)  # same structure, but fully independent
print("c is a:", c is a)
print("c[2] is b:", c[2] is b)  # different object

## 6. Function Parameters as References (Call by Sharing)

Python uses **call by sharing**: each parameter gets a copy of the *reference*. So:
- Functions **can mutate** mutable objects received as arguments.
- Functions **cannot rebind** the caller's variables (rebinding only affects the local name).

In [ ]:
def f(a, b):
    a += b
    return a

# With immutable (int) -- no side effect
x, y = 1, 2
result = f(x, y)
print(f"int:   f({x}, {y}) -> {result},  x={x}, y={y}")

# With mutable (list) -- side effect!
a_list = [1, 2]
b_list = [3, 4]
result = f(a_list, b_list)
print(f"list:  f(...) -> {result},  a_list={a_list}, b_list={b_list}")
# a_list was mutated because += on list does extend in-place

# With immutable (tuple) -- no side effect
t, u = (10, 20), (30, 40)
result = f(t, u)
print(f"tuple: f(...) -> {result},  t={t}, u={u}")

## 7. Mutable Defaults: A Classic Python Trap

Default values are evaluated **once** -- at function definition time. If the default is mutable, all calls that use it **share the same object**.

In [ ]:
# THE BUG: mutable default
class HauntedBus:
    """Ghost passengers appear across instances."""
    def __init__(self, passengers=[]):   # <-- BAD: shared mutable default
        self.passengers = passengers

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

bus1 = HauntedBus(["Alice", "Bill"])
bus1.pick("Charlie")
print("bus1:", bus1.passengers)  # fine

bus2 = HauntedBus()               # uses default []
bus2.pick("Carrie")
print("bus2:", bus2.passengers)  # ["Carrie"]

bus3 = HauntedBus()               # uses THE SAME default []
print("bus3:", bus3.passengers)  # ["Carrie"] -- ghost passenger!

bus3.pick("Dave")
print("bus2:", bus2.passengers)  # ["Carrie", "Dave"] -- bus2 sees it!

print("\nbus2.passengers is bus3.passengers:", bus2.passengers is bus3.passengers)
print("The shared default:", HauntedBus.__init__.__defaults__)

In [ ]:
# THE FIX: use None as sentinel, create fresh list inside
class GoodBus:
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = list(passengers)  # COPY the argument

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

gb1 = GoodBus()
gb1.pick("Carrie")
gb2 = GoodBus()
print("gb1:", gb1.passengers)  # ["Carrie"]
print("gb2:", gb2.passengers)  # [] -- independent!
print("gb1.passengers is gb2.passengers:", gb1.passengers is gb2.passengers)

## 8. Defensive Programming with Mutable Parameters

Even with `None` defaults, aliasing the caller's mutable argument is dangerous.

In [ ]:
# BAD: aliasing the caller's list
class TwilightBus:
    """Passengers vanish from the original roster."""
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = passengers  # <-- alias, not copy!

    def drop(self, name):
        self.passengers.remove(name)

team = ["Sue", "Tina", "Maya", "Diana", "Pat"]
bus = TwilightBus(team)
bus.drop("Tina")
bus.drop("Pat")
print("team after bus drops:", team)  # Sue, Maya, Diana -- Tina & Pat gone!

# FIX: self.passengers = list(passengers)  # make a copy

## 9. `del` and Garbage Collection

- `del` removes a **reference** (name), not the object itself.
- CPython uses **reference counting**: when refcount hits 0, the object is destroyed immediately.
- A cyclic garbage collector handles reference cycles.

In [ ]:
import weakref

# Create object and two references
s1 = {1, 2, 3}
s2 = s1

def farewell():
    print("  ...like tears in the rain.")

ender = weakref.finalize(s1, farewell)
print("alive:", ender.alive)

del s1                        # remove one reference
print("after del s1, alive:", ender.alive)  # still alive (s2 holds it)

s2 = "spam"                   # remove last reference -> object destroyed
print("after s2 rebind, alive:", ender.alive)

## 10. Tricks Python Plays with Immutables

CPython interns small integers and some strings for performance. This means separately-created immutable objects may share identity -- but **never rely on this behavior**.

In [ ]:
# tuple() and [:] on a tuple return the SAME object (optimization)
t1 = (1, 2, 3)
t2 = tuple(t1)
t3 = t1[:]
print("tuple(t1) is t1:", t2 is t1)  # True!
print("t1[:] is t1:    ", t3 is t1)  # True!

# String interning
s1 = "hello"
s2 = "hello"
print("\nstring interning: s1 is s2:", s1 is s2)  # True (interned)

# Small integer caching (-5 to 256 in CPython)
a = 256
b = 256
print("256 is 256:", a is b)   # True (cached)

# BUT: never rely on this! Always use == for value comparison
# The ranges and rules are implementation details that can change.
print("\nAlways use == for comparison:", a == b)

---
## Challenge: Predict the Output

Before running each cell below, **predict** what will be printed. Then run it and check your understanding.

In [ ]:
# Challenge 1: What does this print?
a = [1, 2, 3]
b = a
c = list(a)

a.append(4)

print("b:", b)
print("c:", c)
print("b is a:", b is a)
print("c is a:", c is a)

In [ ]:
# Challenge 2: What does this print?
def add_item(item, target=[]):
    target.append(item)
    return target

print(add_item("a"))
print(add_item("b"))
print(add_item("c", []))
print(add_item("d"))

In [ ]:
# Challenge 3: What does this print?
x = (1, [2, 3])
try:
    x[1] += [4, 5]
except TypeError:
    pass
print("x:", x)
# Hint: both the mutation AND the exception happen!

---
## Key Takeaways

1. **Variables are labels**, not boxes -- assignment binds a name to an object.
2. **`==` tests equality** (values); **`is` tests identity** (same object). Use `is` only for `None` and sentinels.
3. **Tuples are "relatively" immutable** -- their references are fixed, but referred mutable objects can change.
4. **Copies are shallow by default** -- use `copy.deepcopy()` when you need full independence.
5. **Call by sharing**: functions get copies of references; they can mutate mutable args but cannot rebind the caller's variables.
6. **Never use mutable defaults** -- use `None` and create fresh objects inside the function.
7. **Copy arguments defensively** when your class stores them to avoid aliasing the caller's data.
8. **`del` removes references**, not objects; garbage collection handles the rest.
9. **Never rely on interning** -- always use `==` for value comparisons.